# F1 Data Exploration and Model Training

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib
import os

# Create directories if they don't exist
os.makedirs('../ml_data', exist_ok=True)
os.makedirs('../backend/app/models', exist_ok=True)

In [ ]:
# Define file paths (assuming they are in ml_data)
RACES_PATH = '../ml_data/races.csv'
RESULTS_PATH = '../ml_data/results.csv'
DRIVERS_PATH = '../ml_data/drivers.csv'
CONSTRUCTORS_PATH = '../ml_data/constructors.csv'

# Create dummy files for now, as we can't download them
pd.DataFrame().to_csv(RACES_PATH)
pd.DataFrame({
    'resultId': [1, 2, 3, 4, 5],
    'raceId': [1, 1, 1, 1, 1],
    'driverId': [1, 2, 3, 4, 5],
    'constructorId': [1, 2, 3, 4, 5],
    'grid': [1, 2, 3, 4, 5],
    'position': [1, 2, 3, 4, 5]
}).to_csv(RESULTS_PATH, index=False)

pd.DataFrame({
    'raceId': [1],
    'year': [2023],
    'circuitId': [1]
}).to_csv(RACES_PATH, index=False)

pd.DataFrame({
    'driverId': [1, 2, 3, 4, 5],
    'driverRef': ['hamilton', 'bottas', 'verstappen', 'perez', 'leclerc']
}).to_csv(DRIVERS_PATH, index=False)

pd.DataFrame({
    'constructorId': [1, 2, 3, 4, 5],
    'constructorRef': ['mercedes', 'red_bull', 'ferrari', 'mclaren', 'alpine']
}).to_csv(CONSTRUCTORS_PATH, index=False)

In [ ]:
races = pd.read_csv(RACES_PATH)
results = pd.read_csv(RESULTS_PATH)
drivers = pd.read_csv(DRIVERS_PATH)
constructors = pd.read_csv(CONSTRUCTORS_PATH)

## 2. Merge and Clean Data

In [ ]:
df = pd.merge(results, races[['raceId', 'year', 'circuitId']], on='raceId', how='left')
df = pd.merge(df, drivers[['driverId', 'driverRef']], on='driverId', how='left')
df = pd.merge(df, constructors[['constructorId', 'constructorRef']], on='constructorId', how='left')

df = df[['year', 'circuitId', 'driverId', 'constructorId', 'grid', 'position']]

In [ ]:
# Handle non-numeric or missing positions
df['position'] = pd.to_numeric(df['position'], errors='coerce')
df.dropna(subset=['position'], inplace=True)
df['position'] = df['position'].astype(int)

## 3. Define Target and Features

In [ ]:
df['top3_finish'] = df['position'].apply(lambda x: 1 if x <= 3 else 0)
df.drop('position', axis=1, inplace=True)

In [ ]:
features = ['year', 'circuitId', 'driverId', 'constructorId', 'grid']
target = 'top3_finish'

X = df[features]
y = df[target]

## 4. Encode Categorical Variables

In [ ]:
encoders = {}
for col in ['circuitId', 'driverId', 'constructorId']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

## 5. Train Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

## 6. Evaluate Model

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

## 7. Save Artifacts

In [ ]:
joblib.dump(model, '../backend/app/models/f1_model.pkl')
joblib.dump(encoders, '../backend/app/models/encoders.pkl')